# Entity Analysis: Trends, Associations, and Networks

This notebook analyses the entity dataset built in `09_entity_dataset.ipynb`. It loads the saved tables (per-article entities, frequencies, timelines, and co-occurrence) and turns them into interpretable analysis: which diseases and chemicals dominate, what is rising or falling, which drug-disease associations are strongest, and how diseases cluster into a co-occurrence network.

It reads from `data/3_entities/`, so it never touches the 3 million abstracts. It works on whichever method's tables are present (dictionary always, model if notebook 09's model pass was run); set METHOD below to choose.

## Setup: load the saved entity tables

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

ROOT = os.path.dirname(os.getcwd())
ENT_DIR = os.path.join(ROOT, "data", "3_entities")

# choose which method's tables to analyse: "dictionary" (always) or a model tag like "model_full" / "model_50000"
METHOD = "dictionary"

def load(name):
    path = os.path.join(ENT_DIR, f"{name}_{METHOD}.parquet")
    if not os.path.exists(path):
        print(f"missing: {path}")
        return None
    return pd.read_parquet(path)

disease_freq = load("disease_frequency")
chemical_freq = load("chemical_frequency")
disease_tl = load("disease_timeline")
chemical_tl = load("chemical_timeline")
dc_pairs = load("disease_chemical_pairs")
dd_pairs = load("disease_disease_pairs")

# per-article table (filename pattern differs slightly)
_art = os.path.join(ENT_DIR, f"article_entities_{'dictionary' if METHOD=='dictionary' else METHOD}.parquet")
article_df = pd.read_parquet(_art) if os.path.exists(_art) else None

print(f"analysing METHOD = {METHOD}")
print(f"  diseases ranked: {len(disease_freq):,}" if disease_freq is not None else "  no disease freq")
print(f"  chemicals ranked: {len(chemical_freq):,}" if chemical_freq is not None else "  no chemical freq")
print(f"  disease-chemical pairs: {len(dc_pairs):,}" if dc_pairs is not None else "")
print(f"  articles: {len(article_df):,}" if article_df is not None else "")

## Cleaning: merge duplicates and drop noise

The model's raw entity lists contain plural duplicates (tumor and tumors), non-disease terms (death, pain, trauma), and non-chemical terms (smoking, alcohol). This step merges simple plurals and drops the obvious non-entities, so the rankings and analysis below reflect real diseases and chemicals. It runs at analysis time, so it cleans existing output without re-running extraction. The drop lists are explicit and adjustable.

In [ ]:
# Clean the loaded tables: merge plural duplicates and drop non-disease/non-chemical noise.
# This runs at analysis time, so it works on existing model output without re-running the model.

NON_DISEASE = {
    "death", "deaths", "pain", "trauma", "bleeding", "fatigue", "toxicity", "psychiatric",
    "disability", "weight loss", "comorbidity", "malignancy", "depressive", "injuries",
    "injury", "complications", "complication", "symptoms", "symptom", "disease", "diseases",
    "disorder", "disorders", "syndrome", "syndromes", "lesion", "lesions", "abnormalities",
}
NON_CHEMICAL = {
    "smoking", "alcohol", "oxygen", "calcium", "sodium", "iron", "atp", "snp", "water",
    "amino acid", "amino acids", "nucleotide", "nucleotides", "protein", "proteins",
    "glucose",   # keep? glucose is borderline; comment out this line to keep it
}

def merge_plurals(freq_df, key):
    """Merge simple plural duplicates (X and Xs -> X) by summing their counts."""
    counts = dict(zip(freq_df[key], freq_df["n_articles"]))
    merged = {}
    for term, c in counts.items():
        base = term[:-1] if (term.endswith("s") and term[:-1] in counts) else term
        merged[base] = merged.get(base, 0) + c
    out = pd.DataFrame(merged.items(), columns=[key, "n_articles"])
    return out.sort_values("n_articles", ascending=False).reset_index(drop=True)

def clean_freq(freq_df, key, drop_set):
    if freq_df is None:
        return None
    f = freq_df[~freq_df[key].isin(drop_set)].copy()
    f = f[f[key].str.len() > 2]                       # drop 1-2 char tokens
    f = f[f[key].str.contains("[a-z]", regex=True)]   # must contain a letter
    f = merge_plurals(f, key)
    return f

if disease_freq is not None:
    _before = len(disease_freq)
    disease_freq = clean_freq(disease_freq, "disease", NON_DISEASE)
    print(f"diseases: {_before:,} -> {len(disease_freq):,} after cleaning + plural merge")
    print("top 15 cleaned diseases:")
    print(disease_freq.head(15).to_string(index=False))

if chemical_freq is not None:
    _before = len(chemical_freq)
    chemical_freq = clean_freq(chemical_freq, "chemical", NON_CHEMICAL)
    print(f"\nchemicals: {_before:,} -> {len(chemical_freq):,} after cleaning")
    print("top 15 cleaned chemicals:")
    print(chemical_freq.head(15).to_string(index=False))

## 1. The most-studied diseases and chemicals

A simple but informative starting point: what the corpus studies most, by article count.

In [ ]:
if disease_freq is not None and chemical_freq is not None:
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    top_d = disease_freq.head(15)[::-1]
    sns.barplot(x="n_articles", y="disease", data=top_d, color="#c0392b", ax=axes[0])
    axes[0].set_title("Most-studied diseases (articles)"); axes[0].set_xlabel("articles"); axes[0].set_ylabel("")
    top_c = chemical_freq.head(15)[::-1]
    sns.barplot(x="n_articles", y="chemical", data=top_c, color="#2471a3", ax=axes[1])
    axes[1].set_title("Most-studied chemicals (articles)"); axes[1].set_xlabel("articles"); axes[1].set_ylabel("")
    plt.tight_layout(); plt.show()

**What this shows:** the headline composition of the corpus's entity attention. The leading diseases and chemicals are the conditions and substances biomedical research centres on, by sheer volume of articles.

## 2. What is rising and what is falling

Using the timeline tables, the trend of each entity is summarised by comparing its early-period to late-period prevalence. This surfaces the fastest risers and decliners.

In [ ]:
if disease_tl is not None:
    years = sorted(disease_tl.index)
    art_per_year = disease_tl.sum(axis=1)   # approx; for share use article counts if available
    early = [y for y in years if y <= years[0] + 2]
    late = [y for y in years if y >= years[-1] - 2]

    def trend_table(tl):
        rate_early = tl.loc[early].sum() / max(len(early), 1)
        rate_late = tl.loc[late].sum() / max(len(late), 1)
        out = pd.DataFrame({"early": rate_early, "late": rate_late})
        out["change"] = out["late"] - out["early"]
        out["ratio"] = (out["late"] + 1) / (out["early"] + 1)
        return out

    dz_trend = trend_table(disease_tl)
    risers = dz_trend[dz_trend["early"] >= 20].sort_values("ratio", ascending=False).head(10)
    fallers = dz_trend[dz_trend["early"] >= 20].sort_values("ratio").head(10)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    sns.barplot(x=risers["ratio"], y=risers.index, color="#27ae60", ax=axes[0])
    axes[0].set_title("Fastest-rising diseases (late/early ratio)"); axes[0].set_xlabel("growth ratio"); axes[0].set_ylabel("")
    sns.barplot(x=fallers["ratio"], y=fallers.index, color="#7f8c8d", ax=axes[1])
    axes[1].set_title("Fastest-declining diseases"); axes[1].set_xlabel("growth ratio"); axes[1].set_ylabel("")
    plt.tight_layout(); plt.show()
    print("top risers:", ", ".join(risers.index[:8]))

**What this shows:** the diseases gaining and losing research attention over the period. Pandemic-related terms typically dominate the risers. Declines are usually gentler (research areas mature rather than vanish), so the falling side reflects relative shifts in attention.

## 3. Strongest drug-disease associations

The disease-chemical co-occurrence table, shown as a heatmap of the most common pairings among the top entities. This surfaces recognisable therapeutic relationships.

In [ ]:
if dc_pairs is not None and disease_freq is not None and chemical_freq is not None:
    top_dz = disease_freq.head(12)["disease"].tolist()
    top_ch = chemical_freq.head(12)["chemical"].tolist()
    sub = dc_pairs[dc_pairs["disease"].isin(top_dz) & dc_pairs["chemical"].isin(top_ch)]
    mat = sub.pivot_table(index="disease", columns="chemical", values="n_articles", fill_value=0)
    mat = mat.reindex(index=[d for d in top_dz if d in mat.index],
                      columns=[c for c in top_ch if c in mat.columns])
    plt.figure(figsize=(12, 9))
    sns.heatmap(mat, cmap="YlOrRd", annot=True, fmt=".0f", cbar_kws={"label": "co-mentioning articles"})
    plt.title("Drug-disease co-occurrence (top entities)")
    plt.xlabel("chemical"); plt.ylabel("disease")
    plt.tight_layout(); plt.show()

    print("strongest associations overall:")
    for _, r in dc_pairs.head(12).iterrows():
        print(f"  {r['n_articles']:>7,}   {r['disease']}  +  {r['chemical']}")

**What this shows:** the heatmap makes therapeutic pairings visible at a glance (a disease and the drugs studied alongside it). Strong cells are well-established relationships (for example diabetes with insulin and glucose). Co-occurrence is association from co-mention, not proof of treatment.

## 4. Disease co-occurrence network (comorbidity map)

The disease-disease pairs form a network: nodes are diseases, edges are co-mention strength. Community detection groups diseases that are frequently studied together, a data-driven comorbidity map. Requires networkx.

In [ ]:
try:
    import networkx as nx
except ImportError:
    print("networkx not installed; run pip install networkx")
    nx = None

if nx is not None and dd_pairs is not None and disease_freq is not None:
    MIN_PAIR = max(50, int(dd_pairs["n_articles"].quantile(0.85)))
    top_dz = set(disease_freq.head(40)["disease"])
    G = nx.Graph()
    for _, r in dd_pairs.iterrows():
        if r["disease_a"] in top_dz and r["disease_b"] in top_dz and r["n_articles"] >= MIN_PAIR:
            G.add_edge(r["disease_a"], r["disease_b"], weight=r["n_articles"])
    if G.number_of_nodes():
        comps = sorted(nx.connected_components(G), key=len, reverse=True)
        G = G.subgraph(comps[0]).copy()
        from networkx.algorithms.community import greedy_modularity_communities
        comms = list(greedy_modularity_communities(G, weight="weight"))
        cmap = {n: i for i, com in enumerate(comms) for n in com}
        palette = plt.colormaps["tab10"]
        colors = [palette(cmap[n] % 10) for n in G.nodes()]
        deg = dict(G.degree(weight="weight"))
        sizes = [300 + 0.05 * deg[n] for n in G.nodes()]
        pos = nx.spring_layout(G, k=0.6, seed=42, weight="weight")
        ws = [G[u][v]["weight"] for u, v in G.edges()]; wmax = max(ws) if ws else 1
        widths = [0.3 + 3 * (w / wmax) for w in ws]

        plt.figure(figsize=(14, 11))
        nx.draw_networkx_edges(G, pos, width=widths, alpha=0.2, edge_color="#888")
        nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color=colors, alpha=0.9)
        nx.draw_networkx_labels(G, pos, font_size=8)
        plt.title("Disease co-occurrence network (comorbidity communities)")
        plt.axis("off"); plt.tight_layout(); plt.show()
        print(f"network: {G.number_of_nodes()} diseases, {G.number_of_edges()} edges, {len(comms)} communities")
        for i, com in enumerate(comms[:6]):
            print(f"  community {i+1}: {', '.join(sorted(com)[:8])}")

**What this shows:** diseases cluster into communities that are studied together, a comorbidity map recovered from co-mention. Metabolic conditions, cardiovascular conditions, and others tend to form distinct groups. As always, co-mention is association, not clinical confirmation.

## 5. Entity richness per article

Using the per-article table, how many entities a typical article carries, and how that has changed, a measure of how entity-dense the literature is.

In [ ]:
if article_df is not None:
    dz_col = "dz_dict" if "dz_dict" in article_df else ("dz_model" if "dz_model" in article_df else None)
    if dz_col and "n_diseases" in article_df:
        by_year = article_df.groupby("year").agg(
            mean_diseases=("n_diseases", "mean"),
            mean_chemicals=("n_chemicals", "mean"),
            pct_with_disease=("n_diseases", lambda s: (s > 0).mean() * 100),
        )
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        axes[0].plot(by_year.index, by_year["mean_diseases"], marker="o", label="diseases")
        axes[0].plot(by_year.index, by_year["mean_chemicals"], marker="s", label="chemicals")
        axes[0].set_title("Mean entities per article"); axes[0].set_xlabel("year"); axes[0].legend()
        axes[1].plot(by_year.index, by_year["pct_with_disease"], marker="o", color="#c0392b")
        axes[1].set_title("% of articles mentioning a listed disease"); axes[1].set_xlabel("year"); axes[1].set_ylabel("%")
        plt.tight_layout(); plt.show()

**What this shows:** how entity-dense articles are over time. Rising mean entities can reflect genuinely broader studies or simply better detection coverage. The coverage line (share of articles with a detected disease) is a reminder that these counts are bounded by the term list and the abstract text.

## 7. Disease-family deep dive: cancer subtypes

The generic term "cancer" dominates the disease counts, but the corpus also names specific subtypes (breast, lung, prostate, colorectal cancer, melanoma, leukemia). This section breaks the cancer umbrella into its subtypes and analyses them: which are most studied, how they trend, which drugs associate with each, and how often research uses a specific subtype versus the generic label. The family term is a variable, so the same analysis applies to other families (diabetes, cardiovascular) by changing FAMILY_PATTERNS.

In [ ]:
# Define the family by substring patterns. Cancer is the worked example.
FAMILY_PATTERNS = ("cancer", "carcinoma", "melanoma", "leukemia", "leukaemia",
                   "lymphoma", "sarcoma", "myeloma", "glioma", "neoplasm")
GENERIC_TERMS = {"cancer", "cancers", "tumor", "tumors", "tumour", "tumours",
                 "carcinoma", "neoplasm", "neoplasms", "malignancy", "malignancies"}

if disease_freq is not None:
    fam = disease_freq[disease_freq["disease"].apply(
        lambda d: any(p in d for p in FAMILY_PATTERNS))].copy()
    fam["is_specific"] = ~fam["disease"].isin(GENERIC_TERMS)
    specific = fam[fam["is_specific"]].head(15)

    print(f"cancer-family terms found: {len(fam)}")
    print(f"  generic: {fam[~fam['is_specific']]['disease'].tolist()}")
    print(f"\ntop specific subtypes:")
    print(specific[["disease", "n_articles"]].to_string(index=False))

    if len(specific):
        plt.figure(figsize=(11, 7))
        sns.barplot(x="n_articles", y="disease", data=specific[::-1], color="#8e44ad")
        plt.title("Most-studied cancer subtypes (specific terms only)")
        plt.xlabel("articles"); plt.ylabel("")
        plt.gca().xaxis.set_major_formatter(plt.matplotlib.ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
        plt.tight_layout(); plt.show()

**What this shows:** the composition of cancer research below the generic label. The leading specific subtypes are the cancers that draw the most dedicated study, distinct from the large but undifferentiated "cancer" and "tumor" counts.

In [ ]:
if disease_tl is not None and len(specific):
    sub_terms = [d for d in specific["disease"] if d in disease_tl.columns]
    if sub_terms:
        apy = disease_tl.sum(axis=1)
        plt.figure(figsize=(13, 6))
        for term in sub_terms[:8]:
            s = disease_tl[term].reindex(apy.index, fill_value=0) / apy * 1000
            plt.plot(s.index, s.values, marker="o", markersize=3, label=term)
        plt.title("Cancer subtype prevalence over time (per 1,000 articles)")
        plt.xlabel("year"); plt.ylabel("per 1,000 articles")
        plt.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8, frameon=False)
        plt.tight_layout(); plt.show()

**What this shows:** how attention to each subtype moves over time. Diverging trajectories indicate shifting research focus within oncology, for example a subtype rising as new therapies or screening drive study.

In [ ]:
if dc_pairs is not None and len(specific):
    sub_set = set(specific["disease"])
    sub_drug = dc_pairs[dc_pairs["disease"].isin(sub_set)].copy()
    if len(sub_drug):
        top_drugs = sub_drug.groupby("chemical")["n_articles"].sum().nlargest(10).index
        mat = (sub_drug[sub_drug["chemical"].isin(top_drugs)]
               .pivot_table(index="disease", columns="chemical", values="n_articles", fill_value=0))
        if mat.size:
            plt.figure(figsize=(12, max(4, 0.6 * len(mat))))
            sns.heatmap(mat, cmap="Purples", annot=True, fmt=".0f",
                        cbar_kws={"label": "co-mentioning articles"})
            plt.title("Cancer subtype and drug associations")
            plt.xlabel("chemical"); plt.ylabel("subtype")
            plt.tight_layout(); plt.show()
        print("notable subtype-drug pairs:")
        for _, r in sub_drug.head(12).iterrows():
            print(f"  {r['n_articles']:>6,}   {r['disease']}  +  {r['chemical']}")

**What this shows:** the drugs most associated with each cancer subtype. Recognisable pairings (for example breast cancer with tamoxifen or estrogen) validate the signal, and differences across subtypes reflect their distinct therapeutic landscapes.

In [ ]:
if article_df is not None and disease_tl is not None:
    dz_col = "dz_dict" if "dz_dict" in article_df else ("dz_model" if "dz_model" in article_df else None)
    if dz_col:
        def cancer_kind(terms):
            has_specific = any(any(p in t for p in FAMILY_PATTERNS) and t not in GENERIC_TERMS for t in terms)
            has_generic = any(t in GENERIC_TERMS for t in terms)
            return has_specific, has_generic
        rows = []
        for yr, terms in zip(article_df["year"], article_df[dz_col]):
            sp, ge = cancer_kind(terms)
            if sp or ge:
                rows.append((yr, sp, ge))
        cdf = pd.DataFrame(rows, columns=["year", "specific", "generic"])
        by_year = cdf.groupby("year").agg(
            n=("specific", "size"),
            pct_specific=("specific", lambda s: s.mean() * 100),
        )
        plt.figure(figsize=(12, 5))
        plt.plot(by_year.index, by_year["pct_specific"], marker="o", color="#8e44ad")
        plt.title("Specificity of cancer research over time")
        plt.xlabel("year"); plt.ylabel("% of cancer articles naming a specific subtype")
        plt.tight_layout(); plt.show()
        print("a rising line suggests research naming increasingly specific cancer subtypes")

**What this shows:** the granularity of cancer research language over time, the share of cancer-related articles that name a specific subtype rather than only the generic term. A rising line is a proxy for the field becoming more precise. It is a discourse measure, not a measure of the science itself, and it depends on detection coverage.

## 8. Dictionary versus model: what the model adds

This section loads both the dictionary and the model frequency tables and compares them directly, answering whether the expensive model run added meaningful coverage over the fast dictionary. It runs only if both methods have been produced by notebook 09.

In [ ]:
def load_method(name, method):
    p = os.path.join(ENT_DIR, f"{name}_{method}.parquet")
    return pd.read_parquet(p) if os.path.exists(p) else None

# find an available model tag
_model_tags = sorted(os.path.basename(f).replace("disease_frequency_", "").replace(".parquet", "")
                     for f in glob.glob(os.path.join(ENT_DIR, "disease_frequency_model*.parquet")))
dict_dz = load_method("disease_frequency", "dictionary")
model_dz = load_method("disease_frequency", _model_tags[0]) if _model_tags else None

if dict_dz is not None and model_dz is not None:
    d_set, m_set = set(dict_dz["disease"]), set(model_dz["disease"])
    both = d_set & m_set
    print(f"dictionary diseases: {len(d_set):,}")
    print(f"model diseases:      {len(m_set):,}")
    print(f"in both:             {len(both):,}")
    print(f"model-only:          {len(m_set - d_set):,}  (what the model adds)")
    print(f"dictionary-only:     {len(d_set - m_set):,}")
    print(f"\nmodel finds {len(m_set)/max(len(d_set),1):.0f}x more distinct diseases than the dictionary")

    # overlay timelines for shared top diseases
    dict_tl = load_method("disease_timeline", "dictionary")
    model_tl = load_method("disease_timeline", _model_tags[0])
    if dict_tl is not None and model_tl is not None:
        shared_top = [d for d in dict_dz.head(6)["disease"] if d in model_tl.columns and d in dict_tl.columns]
        if shared_top:
            fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
            for term in shared_top:
                axes[0].plot(dict_tl.index, dict_tl[term], marker="o", markersize=3, label=term)
                axes[1].plot(model_tl.index, model_tl[term], marker="o", markersize=3, label=term)
            axes[0].set_title("Dictionary"); axes[1].set_title("Model")
            for a in axes: a.set_xlabel("year"); a.set_ylabel("articles")
            axes[1].legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8, frameon=False)
            plt.suptitle("Same diseases, both methods (counts differ: model counts mentions, dictionary documents)")
            plt.tight_layout(); plt.show()
else:
    print("need both dictionary and model tables. Run notebook 09 with RUN_MODEL=True to enable this comparison.")

**What this shows:** the concrete payoff of the model run. The model finds far more distinct diseases, and the overlaid timelines show that on shared terms the two methods agree in shape while differing in absolute counts (the model counts mentions, the dictionary counts documents). The model-only count is the coverage the dictionary cannot reach.

## 9. Burst detection: emerging research fronts

Trend ratios show steady risers and fallers; bursts are different, entities that spike sharply in a single year. This finds the steepest year-over-year jump for each entity and flags genuine bursts, surfacing emerging fronts (a pandemic term appearing, a therapy class taking off) that a smooth trend would understate.

In [ ]:
def detect_bursts(tl, growth_thresh=2.0, min_year=2017, min_peak=50):
    out = []
    for col in tl.columns:
        s = tl[col]
        if s.max() < min_peak:
            continue
        best = None
        for i in range(1, len(s)):
            yr = s.index[i]
            if yr < min_year:
                continue
            growth = (s.iloc[i] + 1) / (s.iloc[i - 1] + 1)
            if best is None or growth > best[1]:
                best = (yr, growth, s.iloc[i])
        if best and best[1] >= (1 + growth_thresh):
            out.append({"entity": col, "burst_year": int(best[0]),
                        "value_at_burst": int(best[2]), "growth_x": round(best[1], 1)})
    cols = ["entity", "burst_year", "value_at_burst", "growth_x"]
    return (pd.DataFrame(out).sort_values("growth_x", ascending=False).reset_index(drop=True)
            if out else pd.DataFrame(columns=cols))

if disease_tl is not None:
    dz_bursts = detect_bursts(disease_tl)
    print(f"disease bursts detected: {len(dz_bursts)}")
    print(dz_bursts.head(15).to_string(index=False))
    if chemical_tl is not None:
        ch_bursts = detect_bursts(chemical_tl)
        print(f"\nchemical bursts detected: {len(ch_bursts)}")
        print(ch_bursts.head(10).to_string(index=False))

    if len(dz_bursts):
        plt.figure(figsize=(12, 6))
        top_b = dz_bursts.head(8)
        sns.barplot(x="growth_x", y="entity", data=top_b, hue="burst_year", dodge=False, palette="flare")
        plt.title("Strongest disease bursts (single-year growth)")
        plt.xlabel("growth multiple"); plt.ylabel(""); plt.legend(title="year", fontsize=8)
        plt.tight_layout(); plt.show()

**What this shows:** the entities that surged fastest in a single year, with the year of the surge. A pandemic-era term should appear with a 2020 or 2021 burst. Bursts capture the arrival of new topics, which steady trend ratios blur across the whole period.

## 10. Stratification by journal and team size

The per-article table carries journal and author count, unused so far. This section stratifies entity richness by team size and profiles the most entity-active journals, testing whether larger teams produce broader, more multi-entity research.

In [ ]:
if article_df is not None and "n_authors" in article_df and "n_diseases" in article_df:
    a = article_df.copy()
    a["team"] = pd.cut(a["n_authors"], bins=[0, 3, 10, 1000],
                       labels=["small (1-3)", "medium (4-10)", "large (>10)"])
    team_stats = a.groupby("team", observed=True).agg(
        mean_diseases=("n_diseases", "mean"),
        mean_chemicals=("n_chemicals", "mean"),
        n_articles=("uid", "size"),
    )
    print("entity richness by team size:")
    print(team_stats.to_string())

    fig, ax = plt.subplots(figsize=(10, 5))
    team_stats[["mean_diseases", "mean_chemicals"]].plot(kind="bar", ax=ax,
                                                          color=["#c0392b", "#2471a3"])
    ax.set_title("Mean entities per article by team size")
    ax.set_xlabel(""); ax.set_ylabel("mean entities"); plt.xticks(rotation=0)
    plt.tight_layout(); plt.show()
else:
    print("no n_authors/n_diseases columns; stratification skipped.")

**What this shows:** whether collaboration scale relates to entity breadth. If larger teams show higher mean entities, it is consistent with bigger teams doing more integrative, multi-condition work, though it can also reflect longer or more complex abstracts. It is an association, not a causal claim.

In [ ]:
if article_df is not None and "journal" in article_df and article_df["journal"].astype(str).str.len().gt(0).any():
    a = article_df.copy()
    a = a[a["journal"].astype(str).str.len() > 0]
    jstats = a.groupby("journal").agg(
        n_articles=("uid", "size"),
        mean_diseases=("n_diseases", "mean"),
    )
    jstats = jstats[jstats["n_articles"] >= 50].sort_values("mean_diseases", ascending=False)
    print(f"journals with >=50 articles: {len(jstats)}")
    print("\nmost entity-dense journals (mean diseases per article):")
    print(jstats.head(12).to_string())
else:
    print("no journal column populated; journal profiling skipped.")

**What this shows:** which journals carry the most entity-dense articles, a rough map of where multi-condition or mechanism-heavy research concentrates. Counts are bounded by the term list and abstract text, so this profiles detected-entity density, not journal quality.

## 11. Network evolution: how disease associations change

The comorbidity network earlier pooled all years. This section splits the corpus into time windows and builds a disease co-occurrence network for each, tracking whether the literature is becoming more interconnected (rising density) and how communities shift. Windows are set for the reliable part of the corpus.

In [ ]:
try:
    import networkx as nx
except ImportError:
    nx = None

if nx is not None and article_df is not None:
    dz_col = "dz_dict" if "dz_dict" in article_df else ("dz_model" if "dz_model" in article_df else None)
    if dz_col:
        import itertools, collections
        WINDOWS = [(2015, 2018), (2019, 2021), (2022, 2025)]
        top_dz = set(disease_freq.head(30)["disease"]) if disease_freq is not None else None

        fig, axes = plt.subplots(1, len(WINDOWS), figsize=(6 * len(WINDOWS), 6))
        if len(WINDOWS) == 1: axes = [axes]
        densities = []
        for ax, (y0, y1) in zip(axes, WINDOWS):
            sl = article_df[(article_df["year"] >= y0) & (article_df["year"] <= y1)]
            pairs = collections.Counter()
            for terms in sl[dz_col]:
                ds = sorted(set(t for t in terms if (top_dz is None or t in top_dz)))
                pairs.update(itertools.combinations(ds, 2))
            G = nx.Graph()
            thr = max(5, int(np.quantile(list(pairs.values()), 0.85)) if pairs else 5)
            for (a, b), w in pairs.items():
                if w >= thr:
                    G.add_edge(a, b, weight=w)
            dens = nx.density(G) if G.number_of_nodes() > 1 else 0
            densities.append((f"{y0}-{y1}", G.number_of_nodes(), G.number_of_edges(), round(dens, 3)))
            if G.number_of_nodes():
                pos = nx.spring_layout(G, k=0.8, seed=42, weight="weight")
                deg = dict(G.degree(weight="weight"))
                nx.draw_networkx_edges(G, pos, alpha=0.2, edge_color="#888", ax=ax)
                nx.draw_networkx_nodes(G, pos, node_size=[60 + 0.04 * deg[n] for n in G.nodes()],
                                       node_color="#16a085", alpha=0.85, ax=ax)
                nx.draw_networkx_labels(G, pos, font_size=6, ax=ax)
            ax.set_title(f"{y0}-{y1}  (density {dens:.3f})"); ax.axis("off")
        plt.suptitle("Disease co-occurrence network across time windows")
        plt.tight_layout(); plt.show()
        print("window, nodes, edges, density:")
        for d in densities: print("  ", d)
        print("\nrising density across windows would indicate increasingly interconnected disease research")

**What this shows:** how the disease association structure changes over time. Rising network density across windows indicates the literature increasingly links multiple conditions (more multimorbidity research); shifts in which diseases cluster together reveal evolving research framings. As always, these are co-mention associations, exploratory rather than clinical.

## 6. Summary

This notebook turned the saved entity tables into analysis: the most-studied diseases and chemicals, the fastest risers and decliners, a drug-disease association heatmap, a disease comorbidity network, and entity richness over time. All of it reads from `data/3_entities/`, so it is fast and reproducible, and it runs on either the dictionary tables or the model tables by changing METHOD.

### Caveats
Everything here inherits the entity extraction's limits: dictionary recall is a lower bound, the model has higher recall but more noise, co-occurrence is association not causation, and counts are document-level over the filtered, abstract-only corpus. The network and trend views are exploratory, meant to surface patterns for closer study, not to confirm clinical relationships.